# GPT-Style Fantasy Language Model — Fully Annotated Project Notebook

This notebook documents an end-to-end language-model training pipeline built for a graduate final project. It covers dataset collection and cleaning, BPE tokenization, sequence construction, transformer training, validation, checkpointing, and text generation.

## What this notebook is for
- Build a cleaned text corpus from Project Gutenberg-style plain-text books.
- Train a tokenizer for the corpus.
- Convert text into token tensors and create autoregressive training sequences.
- Train a custom GPT-style decoder-only transformer in PyTorch.
- Evaluate the model with validation loss and perplexity.
- Generate fantasy-style text from prompts.

## Recommended run order
Run the notebook top to bottom the first time. After artifacts have been created, later runs can skip data-download and tokenizer-training sections if the files already exist.

## Notes for GitHub readers
- Paths are relative to `Books_Dataset/` so the project can be moved between machines more easily.
- Training is GPU-aware but can still run on CPU at much lower speed.
- Checkpoint files and dataset artifacts are intentionally not embedded in the notebook output.


## Section 1 — Imports
These imports cover file handling, text cleaning, tokenization, model building, training, and progress monitoring.


In [ ]:
# Core imports
import csv
import gc
import glob
import math
import os
import random
import re
import time
import unicodedata
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from bs4 import BeautifulSoup
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup


## Section 2 — Device Detection
This cell checks whether CUDA is available and selects GPU training when possible.


In [ ]:
# Ensuring that torch is using GPU and not CPU otherwise the training will take a very long time
print("GPU available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Section 3 — Global Configuration
This is the central configuration block: dataset folders, saved artifact paths, model hyperparameters, optimizer settings, and training controls all live here.


In [ ]:
# Global settings
SEED = 42
BASE_DIR = Path("Books_Dataset")
ORIGINAL_TEXT_DIR = BASE_DIR / "Original_Text_Downloads"
FORMATTED_TEXT_DIR = BASE_DIR / "Formatted_Original_Text_Files"
CLEANED_TEXT_DIR = BASE_DIR / "Cleaned_Text"
TOKENIZER_PATH = BASE_DIR / "tokenizer.json"
COMBINED_CORPUS_PATH = BASE_DIR / "combined_corpus.txt"
CORPUS_TENSOR_PATH = BASE_DIR / "full_corpus_tokens.pt"
PER_FILE_TENSOR_CACHE_PATH = BASE_DIR / "per_file_token_tensors.pt"
CHECKPOINT_DIR = BASE_DIR / "Saved_Model_and_Checkpoints"
BEST_MODEL_PATH = CHECKPOINT_DIR / "best_model.pth"
TRAINING_LOG_PATH = BASE_DIR / "training_log.csv"
THRESHOLD_MODEL_PATH = CHECKPOINT_DIR / "gpt_fantasy_model_under_threshold.pth"

vocab_size = 16000
bpe_min_frequency = 5
block_size = 256
sequence_stride = 128          # overlap improves sample efficiency
batch_size = 32
dim = 256
num_heads = 4
ff_dim = 4 * dim
num_layers = 4
dropout = 0.1
val_split = 0.1
test_split = 0.1
data_split_strategy = "by_book"   # one of: "by_book", "contiguous", "random"
num_epochs = 10000
lr = 1e-4
weight_decay = 0.02
grad_clip_norm = 1.0
label_smoothing = 0.1
num_workers = 0               # safer for notebooks/Windows
pin_memory = torch.cuda.is_available()
prefetch_factor = None        # only used when num_workers > 0
persistent_workers = False

# Evaluation / stopping behavior
eval_every_n_epochs = 1
val_loss_threshold = 2.3
val_threshold_patience_evals = 500

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## Section 4 — Filename and Download Helpers
These helper functions sanitize filenames and scrape Project Gutenberg metadata so raw books can be downloaded safely.


In [ ]:
# Making sure the filename doesn't have unaccepted symbols
def fix_filename(filename):
    # This line removes all extra symbols from a filename, but keeps apostraphy like "Alice's"
    filename = re.sub(r'[\\/*?:"<>|]', '', filename)
    # I personally hate having spaces in filenames, so I replace them with an underscore _
    filename = filename.replace(" ", "_")
    return filename

# This part is using Beautiful Soup to scrap the gutenberg website to get my books.
# Downloading book and scraping title
def download_book_gutenberg(book_id):
    # Constructing the URLs
    book_url = f"https://www.gutenberg.org/ebooks/{book_id}"
    text_url = f"https://www.gutenberg.org/ebooks/{book_id}.txt.utf-8"

    # Scrape the title from the book's webpage
    response = requests.get(book_url)

    # Checking if the website is responsive to the ping (aka is the website up and running?)
    if response.status_code == 200:

        # This finds the material
        soup = BeautifulSoup(response.content, 'html.parser')

        # This finds the title of the book (there's another title, but that one has so many extra things attached to it)
        # You'd have to look in the HTML code (use 'F12' on the site) to find the proper tag and type
        title_tag = soup.find('td', itemprop = "headline")

        # Defaulting to "None" for title if it can't find a title
        # Problem: will store multiple books under the same name. So, it will only keep the last "None". Needs to be fixed.
        if not title_tag:
            # Fallback to the <title> tag if not found
            title_tag = soup.find('title')

        # .text gets text, .strip removes unwanted elements; .text.strip() gives clean and usable text
        book_title = title_tag.text.strip() if title_tag else f"book_{book_id}"
        
        # Uses fix_filename() to make the name look decent when saving the book
        book_title = fix_filename(book_title)
    else:
        # Gives the filename "None", but there is an issue with this. If multiple books get filed as None, only the last one will be saved.
        # I need to edit this line of code later.
        print(f"Failed to retrieve book information for {book_id}")
        return None

    # Download the actual text content
    response = requests.get(text_url)
    # Checking again if the website is responsive
    if response.status_code == 200:
        with open(f'Books_Dataset/Original_Text_Downloads/{book_title}.txt', 'w', encoding='utf-8') as f:
            f.write(response.text)
        print(f"Book '{book_title}' downloaded successfully!")
    else:
        print(f"Failed to download the text for book {book_id}")

    return book_title

## Section 5 — Optional Book Download Cell
This cell stores the Gutenberg book IDs used for corpus creation. The actual download loop is optional and can stay commented once the books already exist.


In [ ]:
# # 'book_id' is a list of all the id's of books I used to download from gutenberg's website.
# book_id = ['1152', '55', '1251', '10148', '12753', '54', '1252', '831', '3261', '10002', '2892', '8395',
#            '10662', '2885','1557', '13821', '1605', '1267', '1076', '5713', '15948', '8778', '17134', '13820',
#            '84', '2701', '1342', '100', '11', '2641', '37106', '26184', '67979', '2542', '16389', '6761', '75624', 
#           '74753', '74742', '75608', '72890', '1164', '12750', '426', '73181', '64', '26201', '48242', '1377', '20', '35461']

# # I also used other books that I manually downloaded from github and other sources.

# for i in book_id:
#     # Making sure that it only asks for 1 book every 10 seconds so the server is bombarded with pings and download requests 
#     time.sleep(10)
#     download_book_gutenberg(i)

# # I already ran this and and have downloaded all the books already. I will not run this again.

## Section 6 — Text Re-encoding and Cleaning
These functions convert raw files to UTF-8, strip Gutenberg boilerplate, normalize punctuation, and add structure tokens such as chapter markers and book boundaries.


In [ ]:
# Re-encode all downloaded files as clean UTF-8 and standardize the text.
def force_utf8_encoding(source_folder=ORIGINAL_TEXT_DIR, output_folder=FORMATTED_TEXT_DIR):
    """
    Converts every .txt file in source_folder to UTF-8 (without BOM)
    and saves the result one-to-one into output_folder.
    """
    source_folder = Path(source_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    txt_files = sorted(source_folder.glob("*.txt"))
    print(f"Found {len(txt_files)} .txt files in '{source_folder}'.")

    for file_path in txt_files:
        output_path = output_folder / file_path.name
        try:
            content = safe_read_text(file_path)
            if content is None:
                continue
            with output_path.open("w", encoding="utf-8", newline="\n") as outfile:
                outfile.write(content)
        except Exception as e:
            print(f"Failed to re-encode '{file_path.name}': {e}")

    print(f"\nAll files processed and saved to: '{output_folder}'")


def clean_text(text):
    """
    Cleans raw book text while preserving useful structure and punctuation for language modeling.
    """
    if not isinstance(text, str):
        return None

    # Normalize unicode first so equivalent characters share the same representation.
    text = unicodedata.normalize("NFKC", text)

    # Remove common Gutenberg footer/header boilerplate more safely.
    footer_patterns = [
        r"\*\*\* END OF THE PROJECT GUTENBERG EBOOK.*",
        r"End of Project Gutenberg's .*",
    ]
    header_patterns = [
        r"^\*\*\* START OF THE PROJECT GUTENBERG EBOOK.*?\n",
        r"^The Project Gutenberg eBook of .*?\n",
    ]
    for pat in header_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)
    for pat in footer_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)

    # Normalize line endings and common punctuation variants.
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("—", "-").replace("–", "-")
    text = text.replace("\t", " ")

    # Detect chapter-like headings and insert structural markers.
    lines = text.splitlines()
    cleaned_parts = []
    inside_chapter = False
    chapter_buffer = []

    chapter_heading_regex = re.compile(
        r'^\s*(Chapter|Book|Part)\s+(\d+|[IVXLCDM]+)[.:]?\s*$',
        re.IGNORECASE,
    )

    i = 0
    while i < len(lines):
        line = lines[i].strip()
        match = chapter_heading_regex.match(line)

        if match:
            if inside_chapter and chapter_buffer:
                cleaned_parts.append("\n".join(chapter_buffer).strip())
                cleaned_parts.append("<|chapter_end|>")
                chapter_buffer = []

            cleaned_parts.append("<|chapter_start|>")
            cleaned_parts.append(line)
            inside_chapter = True

            j = i + 1
            while j < len(lines) and not lines[j].strip():
                j += 1
            if j < len(lines):
                next_line = lines[j].strip()
                if next_line and not chapter_heading_regex.match(next_line):
                    cleaned_parts.append(next_line)
                    i = j
        else:
            if inside_chapter:
                chapter_buffer.append(lines[i])
            else:
                cleaned_parts.append(lines[i])
        i += 1

    if inside_chapter and chapter_buffer:
        cleaned_parts.append("\n".join(chapter_buffer).strip())
        cleaned_parts.append("<|chapter_end|>")

    text = "\n".join(cleaned_parts)

    # Add a missing space after sentence-ending punctuation when a capital letter follows.
    text = re.sub(r'([A-Za-z])\.([A-Z])', r'\1. \2', text)

    # Preserve chapter markers while filtering only true control characters.
    text = text.replace('<|chapter_start|>', 'CHAPTERSTARTTOKEN')
    text = text.replace('<|chapter_end|>', 'CHAPTERENDTOKEN')

    filtered_chars = []
    for ch in text:
        if ch == "\n":
            filtered_chars.append(ch)
        elif ch.isprintable():
            filtered_chars.append(ch)
        # non-printable control chars are dropped

    text = "".join(filtered_chars)
    text = text.replace('CHAPTERSTARTTOKEN', '<|chapter_start|>')
    text = text.replace('CHAPTERENDTOKEN', '<|chapter_end|>')

    # Normalize whitespace without destroying paragraph structure.
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ ]{2,}', ' ', text)
    text = text.strip()

    return '[Start]\n' + text + '\n[End]\n'


def safe_read_text(file_path):
    """Read a text file with a few common fallback encodings."""
    file_path = Path(file_path)
    encodings = ("utf-8", "cp1252", "latin1")
    for encoding in encodings:
        try:
            with file_path.open("r", encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Failed to read {file_path}: {e}")
            return None
    print(f"Failed to decode {file_path} with utf-8/cp1252/latin1.")
    return None


def remove_escape_characters(text):
    """
    Decode escaped sequences only when the text appears to contain literal escapes.
    This avoids corrupting already-clean text.
    """
    if text is None:
        return None
    if any(marker in text for marker in (r"\n", r"\t", r"\u", r"\x")):
        try:
            return bytes(text, "utf-8").decode("unicode_escape")
        except Exception:
            return text
    return text


def batch_clean_directory(input_dir=FORMATTED_TEXT_DIR, output_dir=CLEANED_TEXT_DIR, log_file=BASE_DIR / 'batch_cleaning_log.txt'):
    """
    Cleans every .txt file in input_dir and saves the result to output_dir.
    A log file records success/failure for every file.
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    log_file = Path(log_file)
    output_dir.mkdir(parents=True, exist_ok=True)
    log_file.parent.mkdir(parents=True, exist_ok=True)

    txt_files = sorted(input_dir.glob("*.txt"))
    print(f"Found {len(txt_files)} .txt files to clean.")

    with log_file.open("w", encoding="utf-8") as log:
        for file_path in txt_files:
            try:
                raw_text = safe_read_text(file_path)
                if raw_text is None:
                    log.write(f"SKIPPED: {file_path.name} (could not decode)\n")
                    continue

                cleaned = clean_text(remove_escape_characters(raw_text))
                if cleaned is None:
                    log.write(f"SKIPPED: {file_path.name} (cleaning returned None)\n")
                    continue

                out_path = output_dir / file_path.name
                with out_path.open("w", encoding="utf-8", newline="\n") as f:
                    f.write(cleaned)
                log.write(f"OK: {file_path.name}\n")
            except Exception as e:
                log.write(f"FAILED: {file_path.name} | {e}\n")

    print(f"Cleaning complete. Cleaned files saved to: {output_dir}")
    print(f"Log saved to: {log_file}")


## Section 7 — Combined Corpus Builder
This helper merges all cleaned books into a single text file for inspection and sanity checking. Training can still use per-book files separately.


In [ ]:
# Combine cleaned text files into one training corpus file.
def combine_cleaned_texts(source_folder=CLEANED_TEXT_DIR, output_path=COMBINED_CORPUS_PATH):
    """
    Combines all cleaned .txt files into one corpus text file.
    Files are kept in deterministic sorted order.
    """
    source_folder = Path(source_folder)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    file_paths = sorted(source_folder.glob("*.txt"))
    with output_path.open("w", encoding="utf-8", newline="\n") as out:
        for file_path in file_paths:
            text = safe_read_text(file_path)
            if text is None:
                continue
            out.write(text.strip() + "\n\n")

    print(f"Combined {len(file_paths)} files into: {output_path}")



# Annotated Pipeline Overview

This notebook builds a **mini GPT-style language model** for fantasy/story generation from raw book text.

## End-to-end flow

1. **Environment + hyperparameters**
   - Import libraries, detect GPU, and define all core paths/hyperparameters.
2. **Data acquisition and formatting**
   - Download books from Project Gutenberg (optional), standardize UTF-8 encoding, and clean the raw text.
3. **Corpus assembly**
   - Merge cleaned books into one inspection corpus while also preserving per-book files for stronger train/validation/test splitting.
4. **Tokenizer training**
   - Train a Byte-Pair Encoding tokenizer on the cleaned corpus and save it to disk.
5. **Token caching + dataset creation**
   - Convert the corpus into token IDs, cache the results, and create overlapping fixed-length training windows.
6. **Model definition**
   - Build a GPT-style transformer with causal self-attention and learned positional embeddings.
7. **Training utilities**
   - Define loss, dataloaders, checkpointing, logging, evaluation, and debugging helpers.
8. **Training run**
   - Initialize the model and optimizer/schedulers, then train while saving checkpoints and the current best model.
9. **Generation**
   - Reload the best model and sample new text from a prompt.

## Recommended run order

Run the notebook top-to-bottom the first time. After the artifacts already exist, the expensive parts (tokenizer training, token caching, etc.) should reuse saved files unless you explicitly retrain/regenerate them.


## Section 8 — Tokenizer Training
This function trains the tokenizer used by the language model. It learns subword units from the cleaned text and saves the tokenizer for reuse.


In [ ]:
def train_bpe_tokenizer(data_dir=CLEANED_TEXT_DIR, vocab_size=vocab_size,
                        min_frequency=bpe_min_frequency, save_path=TOKENIZER_PATH,
                        force_retrain=False):
    """
    Trains a Byte-Pair Encoding tokenizer on all cleaned text files and saves it.
    Uses ByteLevel pre-tokenization so punctuation and spacing patterns are preserved
    more faithfully than a plain whitespace tokenizer.
    """
    data_dir = Path(data_dir)
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    txt_files = sorted(str(p) for p in data_dir.glob("*.txt"))
    if not txt_files:
        raise FileNotFoundError(f"No .txt files found in {data_dir}")

    if save_path.exists() and not force_retrain:
        print(f"Tokenizer already exists at {save_path}. Skipping retraining.")
        return

    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
    tokenizer.decoder = ByteLevelDecoder()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        special_tokens=["<pad>", "<unk>", "[Start]", "[End]", "<|chapter_start|>", "<|chapter_end|>"]
    )

    tokenizer.train(txt_files, trainer)
    tokenizer.save(str(save_path))
    print(f"Tokenizer trained and saved to: {save_path}")


## Section 9 — Token Cache / Corpus Object
This class loads the tokenizer, converts text into token IDs, caches both full-corpus and per-book tensors, and exposes utilities used throughout the notebook.


In [ ]:
class Vocab_and_Token_Creation:
    def __init__(self, data_path=CLEANED_TEXT_DIR, tokenizer_path=TOKENIZER_PATH,
                 tensor_cache_path=CORPUS_TENSOR_PATH,
                 per_file_cache_path=PER_FILE_TENSOR_CACHE_PATH):
        self.data_path = Path(data_path)
        self.tokenizer_path = Path(tokenizer_path)
        self.tensor_cache_path = Path(tensor_cache_path)
        self.per_file_cache_path = Path(per_file_cache_path)
        self.data_path.mkdir(parents=True, exist_ok=True)
        self.tokenizer = Tokenizer.from_file(str(self.tokenizer_path))
        self.vocab = self.tokenizer.get_vocab()
        self.itos = {idx: token for token, idx in self.vocab.items()}
        self.stoi = {token: idx for idx, token in self.itos.items()}

        self.pad_id = self.tokenizer.token_to_id("<pad>")
        self.unk_id = self.tokenizer.token_to_id("<unk>")
        self.start_id = self.tokenizer.token_to_id("[Start]")
        self.end_id = self.tokenizer.token_to_id("[End]")

    def tokenize_text(self, text):
        return self.tokenizer.encode(text)

    def text_to_indices(self, text):
        return self.tokenize_text(text).ids

    def indices_to_tokens(self, ids):
        return [self.itos.get(int(idx), "<unk>") for idx in ids]

    def get_file_token_tensors(self, force_rebuild=False):
        """
        Returns a list of per-file token tensors so data can be split by book,
        which gives a cleaner validation/test boundary than splitting one long stream.
        """
        if self.per_file_cache_path.exists() and not force_rebuild:
            return torch.load(self.per_file_cache_path, map_location="cpu")

        file_paths = sorted(self.data_path.glob("*.txt"))
        if not file_paths:
            raise FileNotFoundError(f"No cleaned text files found in {self.data_path}")

        payload = []
        for file_path in tqdm(file_paths, desc="Tokenizing files", unit="file"):
            text = safe_read_text(file_path)
            if text is None:
                continue

            ids = self.text_to_indices(text)
            if len(ids) < 2:
                continue

            payload.append({
                "file_name": file_path.name,
                "tensor": torch.tensor(ids, dtype=torch.long),
            })

        self.per_file_cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(payload, self.per_file_cache_path)
        print(f"Saved per-file token tensors to: {self.per_file_cache_path}")
        return payload

    def corpus_to_tensor(self, force_rebuild=False):
        """
        Converts the cleaned corpus into a single 1D torch.LongTensor cache.
        This is still useful for quick inspection even when training uses by-book splits.
        """
        if self.tensor_cache_path.exists() and not force_rebuild:
            return torch.load(self.tensor_cache_path, map_location="cpu")

        file_payload = self.get_file_token_tensors(force_rebuild=force_rebuild)
        all_tensors = [item["tensor"] for item in file_payload]
        if not all_tensors:
            raise ValueError("No tokenized files were available to build the corpus tensor.")

        tensor = torch.cat(all_tensors, dim=0)
        self.tensor_cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(tensor, self.tensor_cache_path)
        print(f"Saved token tensor to: {self.tensor_cache_path}")
        return tensor


## Section 10 — Pipeline Initializer
This helper makes sure the tokenizer exists and then returns a ready-to-use corpus/tokenization object for later dataset creation.


In [ ]:
def initialize_story_generator(data_dir=CLEANED_TEXT_DIR,
                               tokenizer_path=TOKENIZER_PATH,
                               tensor_cache_path=CORPUS_TENSOR_PATH,
                               per_file_cache_path=PER_FILE_TENSOR_CACHE_PATH,
                               vocab_size=vocab_size,
                               min_frequency=bpe_min_frequency,
                               force_retrain=False):
    """
    Ensures the tokenizer exists with the requested vocabulary size,
    then returns a Vocab_and_Token_Creation helper object.
    """
    data_dir = Path(data_dir)
    tokenizer_path = Path(tokenizer_path)
    data_dir.mkdir(parents=True, exist_ok=True)

    needs_training = force_retrain or (not tokenizer_path.exists())
    if tokenizer_path.exists() and not force_retrain:
        existing_tokenizer = Tokenizer.from_file(str(tokenizer_path))
        current_vocab_size = len(existing_tokenizer.get_vocab())
        if current_vocab_size < vocab_size:
            print(f"Existing tokenizer vocab ({current_vocab_size}) is smaller than requested ({vocab_size}). Retraining.")
            needs_training = True

    if needs_training:
        train_bpe_tokenizer(
            data_dir=data_dir,
            vocab_size=vocab_size,
            min_frequency=min_frequency,
            save_path=tokenizer_path,
            force_retrain=True,
        )

    return Vocab_and_Token_Creation(
        data_path=data_dir,
        tokenizer_path=tokenizer_path,
        tensor_cache_path=tensor_cache_path,
        per_file_cache_path=per_file_cache_path,
    )


## Section 11 — Sequence Dataset
This dataset turns one long token stream into overlapping fixed-length training examples for next-token prediction.


In [ ]:
class Tensor_Sequence_Generator(Dataset):
    def __init__(self, token_tensor, block_size, stride=None):
        """
        token_tensor: 1D torch.LongTensor containing the full corpus.
        block_size:  length of each input sequence.
        stride:      step size between windows. Smaller than block_size gives overlap.
        """
        self.token_tensor = token_tensor
        self.block_size = int(block_size)
        self.stride = int(stride) if stride is not None else int(block_size)

        max_start = len(self.token_tensor) - (self.block_size + 1)
        if max_start < 0:
            raise ValueError("Token tensor is shorter than block_size + 1.")
        self.num_sequences = 1 + (max_start // self.stride)

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, index):
        start = index * self.stride
        end = start + self.block_size + 1
        chunk = self.token_tensor[start:end]

        input_ids = chunk[:-1]
        target_ids = chunk[1:]
        return input_ids, target_ids


## Section 12 — Transformer Model
These cells define the GPT-style architecture: transformer blocks plus the full language model that predicts the next token.


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, ff_dim, dropout):
        super().__init__()
        self.ln_1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attn_dropout = nn.Dropout(dropout)

        self.ln_2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask):
        attn_input = self.ln_1(x)
        attn_out, _ = self.attn(attn_input, attn_input, attn_input, attn_mask=attn_mask, need_weights=False)
        x = x + self.attn_dropout(attn_out)
        x = x + self.ff(self.ln_2(x))
        return x


class GPTModel(nn.Module):
    def __init__(self, vocab_size, block_size, dim=512, num_heads=8,
                 ff_dim=1024, num_layers=6, dropout=0.2):
        super().__init__()
        self.block_size = block_size
        self.dim = dim

        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.position_embedding = nn.Embedding(block_size, dim)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim=dim, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # Weight tying improves parameter efficiency and often helps language modeling.
        self.lm_head.weight = self.token_embedding.weight

        causal_mask = torch.triu(torch.full((block_size, block_size), float("-inf")), diagonal=1)
        self.register_buffer("causal_mask", causal_mask, persistent=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T = x.shape
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.block_size}.")

        positions = torch.arange(0, T, device=x.device).unsqueeze(0)
        h = self.token_embedding(x) + self.position_embedding(positions)
        h = self.dropout(h)

        attn_mask = self.causal_mask[:T, :T]
        for block in self.blocks:
            h = block(h, attn_mask=attn_mask)

        h = self.final_norm(h)
        logits = self.lm_head(h)
        return logits


## Section 13 — Loss Function
This custom label-smoothed cross-entropy is the training objective used for both optimization and evaluation.


In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1, ignore_index=-100):
        super(LabelSmoothingCrossEntropy, self).__init__()
        assert 0.0 <= smoothing < 1.0
        self.smoothing = smoothing
        self.ignore_index = ignore_index

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=-1)  # Apply log_softmax to get log-probabilities
        n_classes = pred.size(-1)

        # Create a mask to ignore padding tokens
        ignore = target == self.ignore_index
        target = target.clone()
        target[ignore] = 0  # Temporarily set ignored targets to class 0

        # Create the smoothed labels
        true_dist = torch.zeros_like(pred)
        true_dist.fill_(self.smoothing / (n_classes - 1))
        true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        true_dist.masked_fill_(ignore.unsqueeze(1), 0.0)

        # Compute KL-divergence between prediction and smoothed target
        loss = torch.sum(-true_dist * pred, dim=-1)
        loss = loss.masked_fill(ignore, 0.0)

        return loss.sum() / (~ignore).sum()

## Section 14 — DataLoaders and Splitting
This function builds train/validation/test datasets and loaders. The split strategy controls whether the split is contiguous, random, or by-book.


In [ ]:
def get_dataloaders(generator, block_size=block_size, batch_size=batch_size,
                    val_split=val_split, test_split=test_split,
                    stride=sequence_stride, split_strategy=data_split_strategy,
                    num_workers=num_workers, pin_memory=pin_memory,
                    persistent_workers=persistent_workers):
    """
    Returns DataLoaders for train/validation/test.

    split_strategy:
        - "by_book": split at file boundaries (strongest evaluation split here)
        - "contiguous": split one long token stream into contiguous chunks
        - "random": random sequence split
    """
    if split_strategy not in {"by_book", "contiguous", "random"}:
        raise ValueError("split_strategy must be one of: 'by_book', 'contiguous', 'random'")

    if split_strategy == "by_book":
        file_payload = generator.get_file_token_tensors()
        if len(file_payload) < 3:
            raise ValueError("Need at least 3 cleaned books/files for a by-book train/val/test split.")

        eligible = [item for item in file_payload if len(item["tensor"]) >= block_size + 1]
        if len(eligible) < 3:
            raise ValueError("Need at least 3 sufficiently long files for a by-book split.")

        rng = random.Random(SEED)
        shuffled = eligible[:]
        rng.shuffle(shuffled)

        total_files = len(shuffled)
        val_files = max(1, int(round(total_files * val_split)))
        test_files = max(1, int(round(total_files * test_split)))
        train_files = total_files - val_files - test_files
        if train_files <= 0:
            raise ValueError("Not enough files for requested train/val/test book split.")

        train_items = shuffled[:train_files]
        val_items = shuffled[train_files:train_files + val_files]
        test_items = shuffled[train_files + val_files:]

        def build_concat_dataset(items):
            datasets = [
                Tensor_Sequence_Generator(item["tensor"], block_size, stride=stride)
                for item in items
            ]
            return ConcatDataset(datasets)

        train_dataset = build_concat_dataset(train_items)
        val_dataset = build_concat_dataset(val_items)
        test_dataset = build_concat_dataset(test_items)

        print(f"Split by book | Train books: {len(train_items)} | Val books: {len(val_items)} | Test books: {len(test_items)}")
        print("Train files sample:", [item["file_name"] for item in train_items[:5]])
        print("Val files sample:  ", [item["file_name"] for item in val_items[:5]])
        print("Test files sample: ", [item["file_name"] for item in test_items[:5]])

    else:
        corpus_tensor = generator.corpus_to_tensor()
        full_dataset = Tensor_Sequence_Generator(corpus_tensor, block_size, stride=stride)

        total = len(full_dataset)
        val_size = int(total * val_split)
        test_size = int(total * test_split)
        train_size = total - val_size - test_size

        if train_size <= 0 or val_size <= 0 or test_size <= 0:
            raise ValueError("Dataset split sizes are invalid. Reduce val/test split or use more data.")

        if split_strategy == "contiguous":
            train_indices = list(range(0, train_size))
            val_indices = list(range(train_size, train_size + val_size))
            test_indices = list(range(train_size + val_size, total))
        else:
            all_indices = torch.randperm(total, generator=torch.Generator().manual_seed(SEED)).tolist()
            train_indices = all_indices[:train_size]
            val_indices = all_indices[train_size:train_size + val_size]
            test_indices = all_indices[train_size + val_size:]

        train_dataset = Subset(full_dataset, train_indices)
        val_dataset = Subset(full_dataset, val_indices)
        test_dataset = Subset(full_dataset, test_indices)

    loader_kwargs = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    if num_workers > 0:
        loader_kwargs["persistent_workers"] = persistent_workers
        if prefetch_factor is not None:
            loader_kwargs["prefetch_factor"] = prefetch_factor

    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

    print(f"Train sequences: {len(train_dataset):,}")
    print(f"Val sequences:   {len(val_dataset):,}")
    print(f"Test sequences:  {len(test_dataset):,}")

    return train_loader, val_loader, test_loader


## Section 15 — Checkpoint Saving
This helper saves training state so long runs can be resumed and so the best models can be recovered later.


In [ ]:
def save_checkpoint(model, optimizer, epoch, train_loss, val_loss=None,
                    save_dir=CHECKPOINT_DIR, max_checkpoints=5,
                    plateau_scheduler=None, warmup_scheduler=None,
                    scaler=None, extra_state=None):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = save_dir / f'checkpoint_epoch_{epoch}.pth'
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'learning_rate': optimizer.param_groups[0]['lr'],
    }
    if val_loss is not None:
        checkpoint['val_loss'] = val_loss
    if plateau_scheduler is not None:
        checkpoint['plateau_scheduler_state_dict'] = plateau_scheduler.state_dict()
    if warmup_scheduler is not None:
        checkpoint['warmup_scheduler_state_dict'] = warmup_scheduler.state_dict()
    if scaler is not None:
        checkpoint['scaler_state_dict'] = scaler.state_dict()
    if extra_state is not None:
        checkpoint['extra_state'] = extra_state

    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved at {checkpoint_path}")

    checkpoints = sorted(
        save_dir.glob('checkpoint_epoch_*.pth'),
        key=lambda p: int(p.stem.split('_')[-1])
    )
    while len(checkpoints) > max_checkpoints:
        to_delete = checkpoints.pop(0)
        to_delete.unlink(missing_ok=True)
        print(f"Deleted old checkpoint: {to_delete.name}")


def load_checkpoint(model, optimizer, save_dir=CHECKPOINT_DIR,
                    plateau_scheduler=None, warmup_scheduler=None,
                    scaler=None, map_location=device):
    save_dir = Path(save_dir)
    checkpoints = sorted(
        save_dir.glob('checkpoint_epoch_*.pth'),
        key=lambda p: int(p.stem.split('_')[-1])
    ) if save_dir.exists() else []

    if not checkpoints:
        print("No checkpoint found. Starting from scratch.")
        return 0, None, False, {}

    path = checkpoints[-1]
    checkpoint = torch.load(path, map_location=map_location)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if plateau_scheduler is not None and 'plateau_scheduler_state_dict' in checkpoint:
        plateau_scheduler.load_state_dict(checkpoint['plateau_scheduler_state_dict'])
    if warmup_scheduler is not None and 'warmup_scheduler_state_dict' in checkpoint:
        warmup_scheduler.load_state_dict(checkpoint['warmup_scheduler_state_dict'])
    if scaler is not None and 'scaler_state_dict' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    epoch = checkpoint['epoch']
    last_lr = checkpoint.get('learning_rate', optimizer.param_groups[0]['lr'])
    extra_state = checkpoint.get('extra_state', {})

    print(
        f"Checkpoint loaded: {path} | Epoch {epoch} | "
        f"Train Loss: {checkpoint.get('train_loss', float('nan')):.4f} | "
        f"Val Loss: {checkpoint.get('val_loss', float('nan')):.4f}"
    )
    return epoch, last_lr, True, extra_state


def save_best_model(model, optimizer, epoch, train_loss, val_loss,
                    path=BEST_MODEL_PATH, metadata=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    payload = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'metadata': metadata or {},
    }

    if path.exists():
        checkpoint = torch.load(path, map_location='cpu')
        if val_loss >= checkpoint.get('val_loss', float('inf')):
            return "Kept previous best model"

    torch.save(payload, path)
    return "Saved / updated best model"


## Section 16 — CSV Logging
Each epoch’s metrics are appended to a log file so training progress can be analyzed later without re-running the notebook.


In [ ]:
def log_training_progress(epoch, train_loss, val_loss, epoch_time, lr,
                          train_perplexity=None, val_perplexity=None,
                          grad_norm=None, log_file=TRAINING_LOG_PATH):
    """Append one epoch of training statistics to a CSV log."""
    log_file = Path(log_file)
    log_file.parent.mkdir(parents=True, exist_ok=True)
    log_exists = log_file.exists()

    with log_file.open(mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not log_exists:
            writer.writerow([
                "Epoch", "Train_Loss", "Val_Loss", "Train_Perplexity", "Val_Perplexity",
                "Epoch_Time_Min", "Learning_Rate", "Grad_Norm", "Timestamp"
            ])
        writer.writerow([
            epoch,
            round(train_loss, 6),
            round(val_loss, 6),
            round(train_perplexity, 4) if train_perplexity is not None else "",
            round(val_perplexity, 4) if val_perplexity is not None else "",
            round(epoch_time / 60, 2),
            lr,
            round(grad_norm, 4) if grad_norm is not None else "",
            datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        ])


## Section 17 — Inspection Utilities
These helpers let you inspect tokenized data, corpus size, and sequence windows so you can verify preprocessing before training.


In [ ]:
def inspect_corpus(generator, block_size=block_size, stride=sequence_stride):
    """Print high-level stats about the tokenized corpus."""
    corpus_tensor = generator.corpus_to_tensor()
    total_tokens = len(corpus_tensor)
    num_sequences = len(Tensor_Sequence_Generator(corpus_tensor, block_size, stride=stride))

    print(f"Total tokens:        {total_tokens:,}")
    print(f"Block size:          {block_size}")
    print(f"Stride:              {stride}")
    print(f"Sequences (chunks):  {num_sequences:,}")
    print(f"Approx tensor size:  {corpus_tensor.element_size() * corpus_tensor.numel() / (1024 * 1024):.2f} MB")
    return total_tokens, num_sequences


def show_token_sequence(generator, block_size=block_size, index=0, stride=sequence_stride):
    dataset = Tensor_Sequence_Generator(generator.corpus_to_tensor(), block_size, stride=stride)
    x, y = dataset[index]
    print("Input IDs:", x.tolist()[:25], "...")
    print("Target IDs:", y.tolist()[:25], "...")
    print("Decoded input preview:")
    print(generator.tokenizer.decode(x.tolist()[:100]))


## Section 18 — Token Inspection
This cell shows how the tokenizer maps early corpus pieces to token IDs, which is useful for debugging vocabulary behavior.


In [ ]:
# This block of code checks how words are tokenized. This uses the corpus_to_tensor() function from the previous block.
def show_first_tokens_with_ids(generator, num_tokens=50):
    """
    Displays the first `num_tokens` from the full corpus with their corresponding token IDs.
    """
    # Load the full corpus
    tensor = generator.corpus_to_tensor()

    # Take the first `num_tokens` indices
    ids = tensor[:num_tokens].tolist()

    # Convert each index to its corresponding token
    tokens = generator.indices_to_tokens(ids)

    # Print them side by side
    print(f"{'Token':<20} → ID")
    print("-" * 35)
    for token, index in zip(tokens, ids):
        print(f"{token:<20} → {index}")

## Section 19 — Quote Search Utility
This helper searches the cleaned books for a phrase so you can inspect where specific text appears in the corpus.


In [ ]:
def find_quote(quote, data_dir=CLEANED_TEXT_DIR, context_window=150):
    """
    Search all cleaned text files for a quote and print surrounding context.
    """
    quote = quote.strip()
    quote_lower = quote.lower()
    file_paths = sorted(Path(data_dir).glob('*.txt'))

    matches = 0
    for file_path in file_paths:
        text = safe_read_text(file_path)
        if text is None:
            continue
        text_lower = text.lower()
        start = text_lower.find(quote_lower)
        if start == -1:
            continue

        matches += 1
        end = start + len(quote)
        snippet_start = max(0, start - context_window)
        snippet_end = min(len(text), end + context_window)
        before = text[snippet_start:start]
        highlight = f"||||| {text[start:end]} |||||"
        after = text[end:snippet_end]

        print(f"\nQuote found in: {file_path}")
        print(f"...{before}{highlight}{after}...")

    if matches == 0:
        print("Quote not found in any file.")
    else:
        print(f"\nFound {matches} file(s) containing '{quote}'.")


## Section 20 — Run the UTF-8 Formatter
This executes the re-encoding step on the raw downloaded books.


In [ ]:
# Executing the function
force_utf8_encoding()

## Section 21 — Run the Cleaner
This executes the batch text cleaning pipeline and writes cleaned books to the cleaned-text directory.


In [ ]:
# Setting input/output for file cleaning
input_dir = 'Books_Dataset/Formatted_Original_Text_Files'
output_dir = 'Books_Dataset/Cleaned_Text'
log_file = 'Books_Dataset/batch_cleaning_log.txt'

# Run the batch cleaner
batch_clean_directory(input_dir, output_dir, log_file)

## Section 22 — Build and Inspect the Combined Corpus
This creates the merged corpus file and prints coarse statistics such as total character count.


In [ ]:
# Create the full combined corpus text file.
combine_cleaned_texts()

combined_corpus_text = safe_read_text(COMBINED_CORPUS_PATH)
if combined_corpus_text is None:
    raise FileNotFoundError(f"Could not read combined corpus at {COMBINED_CORPUS_PATH}")

print("Length of dataset in characters:", len(combined_corpus_text))


## Section 23 — Initialize Tokenization and DataLoaders
This is the main setup cell that builds the corpus object and creates train/validation/test loaders.


In [ ]:
# Main pipeline setup.
# - initialize_story_generator(...) ensures the tokenizer exists and prepares cached token tensors.
# - get_dataloaders(...) builds train/validation/test loaders from the tokenized corpus.
# This is the first cell that makes the project training-ready.

generator = initialize_story_generator(
    data_dir=CLEANED_TEXT_DIR,
    tokenizer_path=TOKENIZER_PATH,
    tensor_cache_path=CORPUS_TENSOR_PATH,
    vocab_size=vocab_size,
    min_frequency=bpe_min_frequency,
)

train_loader, val_loader, test_loader = get_dataloaders(
    generator,
    block_size=block_size,
    batch_size=batch_size,
    val_split=val_split,
    test_split=test_split,
    stride=sequence_stride,
    split_strategy=data_split_strategy,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
)

for input_batch, target_batch in train_loader:
    print("Input shape:", input_batch.shape)
    print("Target shape:", target_batch.shape)
    break


## Section 24 — Corpus Tensor Sanity Check
A quick shape check to confirm that tokenization produced the expected tensor.


In [ ]:
corpus_tensor = generator.corpus_to_tensor()
print("Final tensor shape:", corpus_tensor.shape)

## Section 25 — Token Frequency Export
This analyzes token frequencies and writes them to CSV so the learned vocabulary can be inspected outside the notebook.


In [ ]:
# Export token frequencies so the tokenizer vocabulary can be inspected easily.
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
file_paths = sorted(CLEANED_TEXT_DIR.glob("*.txt"))
output_csv_path = BASE_DIR / "token_frequencies.csv"

token_counter = Counter()
for path in tqdm(file_paths, desc="Counting token frequencies", unit="file"):
    text = safe_read_text(path)
    if text is None:
        continue
    token_counter.update(tokenizer.encode(text).tokens)

sorted_tokens = token_counter.most_common()
output_csv_path.parent.mkdir(parents=True, exist_ok=True)
with output_csv_path.open("w", newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Token", "Frequency"])
    writer.writerows(sorted_tokens)

print(f"Token frequency CSV saved to: {output_csv_path}")


## Section 26 — Rebuild DataLoaders (Optional)
Use this if you change split or batching hyperparameters and want to rebuild the loaders without redoing the entire pipeline.


In [ ]:
# Rebuild data loaders if you want a fresh split after changing hyperparameters.
train_loader, val_loader, test_loader = get_dataloaders(
    generator,
    block_size=block_size,
    batch_size=batch_size,
    val_split=val_split,
    test_split=test_split,
    stride=sequence_stride,
    split_strategy=data_split_strategy,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
)


## Section 27 — Model Initialization
This constructs the GPT-style model and prints the parameter count so model scale is visible immediately.


In [ ]:
# Initializing the model
model = GPTModel(
    vocab_size=vocab_size,
    block_size=block_size,
    dim=dim,
    num_heads=num_heads,
    ff_dim=ff_dim,
    num_layers=num_layers,
    dropout=dropout
).to("cuda" if torch.cuda.is_available() else "cpu")

# Seeing how many parameters I am training... o.o
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")

## Section 28 — Early Token Debug View
A compact token/ID sanity check.


In [ ]:
show_first_tokens_with_ids(generator, num_tokens=20)

## Section 29 — Corpus / Sequence Debug View
These helpers print overall corpus statistics and show an example sequence-target pair.


In [ ]:
inspect_corpus(generator, block_size=512)
show_token_sequence(generator, block_size=256, index=0)

## Section 30 — Manual Vocabulary Inspection
This lets you search a simple quote/word inside the cleaned corpus when checking whether tokenization learned something sensible.


In [ ]:
# Look at your own tokenizer and check to see if any of the words makes no sense. Then look it up here:
find_quote("yes")

## Section 31 — Evaluation Function
This function runs the model on a loader without gradient updates and reports average loss plus perplexity.


In [ ]:
def evaluate(model, data_loader, criterion, device, max_batches=None):
    """
    Evaluate the model using the SAME loss function as training.
    Returns average loss and perplexity.
    """
    model.eval()
    total_loss = 0.0
    batches_seen = 0
    max_batches = max_batches if max_batches is not None else len(data_loader)

    start = time.time()
    with torch.no_grad():
        for batch_idx, (x, y) in enumerate(
            tqdm(data_loader, total=min(len(data_loader), max_batches), desc="Evaluating", unit="batch")
        ):
            if batch_idx >= max_batches:
                break
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            outputs = model(x)
            loss = criterion(outputs.reshape(-1, outputs.size(-1)), y.reshape(-1))
            total_loss += loss.item()
            batches_seen += 1

    avg_loss = total_loss / max(1, batches_seen)
    perplexity = float(math.exp(min(avg_loss, 20.0)))
    duration = int(time.time() - start)
    print(f"Evaluation complete | Avg Loss: {avg_loss:.4f} | PPL: {perplexity:.2f} | Time: {duration//60}m {duration%60}s")
    return avg_loss, perplexity


def train(model, train_loader, val_loader, optimizer, criterion, device,
          num_epochs=500, train_mode="full", target_batches=None,
          early_stop_patience=10,
          val_loss_threshold=val_loss_threshold,
          val_threshold_patience_evals=val_threshold_patience_evals,
          eval_every_n_epochs=eval_every_n_epochs,
          save_dir=CHECKPOINT_DIR,
          best_model_path=BEST_MODEL_PATH,
          threshold_model_path=THRESHOLD_MODEL_PATH,
          max_checkpoints=5,
          warmup_scheduler=None,
          plateau_scheduler=None,
          warmup_steps=None,
          grad_clip_norm=grad_clip_norm,
          use_amp=None,
          eval_max_batches=None):
    """
    Train the GPT model with mixed precision, checkpoint resume, scheduler support,
    gradient clipping, logging, and consistent evaluation.

    Important:
    - val_threshold_patience_evals counts consecutive *validation evaluations* under
      the threshold, not epochs or training batches.
    - checkpoint extra_state is saved *after* counters are updated so resume logic
      stays faithful.
    """
    if target_batches is None:
        target_batches = len(train_loader)
    if eval_every_n_epochs < 1:
        raise ValueError("eval_every_n_epochs must be at least 1.")

    use_amp = torch.cuda.is_available() if use_amp is None else use_amp
    scaler = GradScaler(enabled=use_amp)

    start_epoch, last_lr, resumed, extra_state = load_checkpoint(
        model,
        optimizer,
        save_dir=save_dir,
        plateau_scheduler=plateau_scheduler,
        warmup_scheduler=warmup_scheduler,
        scaler=scaler,
        map_location=device,
    )

    using_warmup = warmup_scheduler is not None
    warmup_counter = 0
    best_val_loss = float('inf')
    no_improve_evals = 0
    low_val_loss_eval_count = 0
    last_val_loss = None
    last_val_perplexity = None

    if resumed:
        if last_lr is not None:
            for group in optimizer.param_groups:
                group['lr'] = last_lr

        best_val_loss = extra_state.get('best_val_loss', best_val_loss)
        no_improve_evals = extra_state.get('no_improve_evals', 0)
        low_val_loss_eval_count = extra_state.get('low_val_loss_eval_count', 0)
        warmup_counter = extra_state.get('warmup_counter', 0)
        last_val_loss = extra_state.get('last_val_loss', None)
        last_val_perplexity = extra_state.get('last_val_perplexity', None)

        if warmup_steps is not None and warmup_counter >= warmup_steps:
            using_warmup = False

        print(f"Resumed training from epoch {start_epoch}. Current LR: {optimizer.param_groups[0]['lr']:.8f}")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        total_loss = 0.0
        batches_processed = 0
        running_grad_norm = None
        epoch_start = time.time()

        progress_bar = tqdm(
            train_loader,
            total=min(len(train_loader), target_batches),
            desc=f"Epoch {epoch + 1}/{num_epochs}",
            unit="batch"
        )

        for batch_idx, (x, y) in enumerate(progress_bar):
            if batch_idx >= target_batches:
                break

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=use_amp):
                outputs = model(x)
                loss = criterion(outputs.reshape(-1, outputs.size(-1)), y.reshape(-1))

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            running_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()

            if using_warmup and warmup_scheduler is not None and warmup_steps is not None and warmup_counter < warmup_steps:
                warmup_scheduler.step()
                warmup_counter += 1
                if warmup_counter >= warmup_steps:
                    using_warmup = False
                    print("Warmup complete. Plateau scheduler is now active.")

            total_loss += loss.item()
            batches_processed += 1
            avg_loss_so_far = total_loss / batches_processed
            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}",
                avg=f"{avg_loss_so_far:.4f}",
                lr=f"{optimizer.param_groups[0]['lr']:.2e}"
            )

        avg_train_loss = total_loss / max(1, batches_processed)
        train_perplexity = float(math.exp(min(avg_train_loss, 20.0)))

        did_evaluate = ((epoch + 1) % eval_every_n_epochs == 0) or (epoch == num_epochs - 1)
        status = "No validation run this epoch"

        if did_evaluate:
            val_loss, val_perplexity = evaluate(model, val_loader, criterion, device, max_batches=eval_max_batches)
            last_val_loss = val_loss
            last_val_perplexity = val_perplexity

            if (not using_warmup) and plateau_scheduler is not None:
                plateau_scheduler.step(val_loss)
                print(f"[Scheduler] LR after epoch {epoch + 1}: {optimizer.param_groups[0]['lr']:.8f}")

            if val_loss < best_val_loss - 1e-6:
                best_val_loss = val_loss
                no_improve_evals = 0
            else:
                no_improve_evals += 1

            if val_loss < val_loss_threshold:
                low_val_loss_eval_count += 1
            else:
                low_val_loss_eval_count = 0

            status = save_best_model(
                model=model,
                optimizer=optimizer,
                epoch=epoch + 1,
                train_loss=avg_train_loss,
                val_loss=val_loss,
                path=best_model_path,
                metadata={
                    'train_perplexity': train_perplexity,
                    'val_perplexity': val_perplexity,
                    'vocab_size': vocab_size,
                    'block_size': block_size,
                    'sequence_stride': sequence_stride,
                    'eval_every_n_epochs': eval_every_n_epochs,
                },
            )

        epoch_time = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]['lr']
        log_training_progress(
            epoch=epoch + 1,
            train_loss=avg_train_loss,
            val_loss=last_val_loss,
            epoch_time=epoch_time,
            lr=current_lr,
            train_perplexity=train_perplexity,
            val_perplexity=last_val_perplexity,
            grad_norm=float(running_grad_norm) if running_grad_norm is not None else None,
        )

        extra_state = {
            'best_val_loss': best_val_loss,
            'no_improve_evals': no_improve_evals,
            'low_val_loss_eval_count': low_val_loss_eval_count,
            'warmup_counter': warmup_counter,
            'last_val_loss': last_val_loss,
            'last_val_perplexity': last_val_perplexity,
        }
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch + 1,
            train_loss=avg_train_loss,
            val_loss=last_val_loss,
            save_dir=save_dir,
            max_checkpoints=max_checkpoints,
            plateau_scheduler=plateau_scheduler,
            warmup_scheduler=warmup_scheduler,
            scaler=scaler,
            extra_state=extra_state,
        )

        print(
            f"Epoch {epoch + 1} complete | Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {last_val_loss if last_val_loss is not None else float('nan'):.4f} | "
            f"Train PPL: {train_perplexity:.2f} | "
            f"Val PPL: {last_val_perplexity if last_val_perplexity is not None else float('nan'):.2f} | "
            f"{status}"
        )

        if did_evaluate and no_improve_evals >= early_stop_patience:
            print("Early stopping: validation loss stopped improving across consecutive evaluations.")
            break

        if did_evaluate and low_val_loss_eval_count >= val_threshold_patience_evals:
            torch.save(
                {
                    'model_state_dict': model.state_dict(),
                    'val_loss': last_val_loss,
                    'metadata': {
                        'threshold': val_loss_threshold,
                        'consecutive_evaluations': low_val_loss_eval_count,
                    },
                },
                threshold_model_path
            )
            print("Early stopping: validation loss stayed under threshold for consecutive evaluations. Model saved.")
            break

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("Training complete!")


## Section 32 — Quick Validation Probe
A lightweight check that runs evaluation on the validation loader before or during training.


In [ ]:

# Quick validation-set evaluation helper.
# This cell is intentionally self-contained so it still works after a kernel restart
# or if the optimizer/scheduler setup cell has not been run yet.
criterion = LabelSmoothingCrossEntropy(
    smoothing=label_smoothing,
    ignore_index=-100,
)

val_loss, val_perplexity = evaluate(model, val_loader, criterion, device)
print(f"Validation Loss: {val_loss:.4f} | Validation Perplexity: {val_perplexity:.2f}")


## Section 33 — Optimizer and Scheduler Setup
This sets up TF32 (if available), the training loss, AdamW optimizer, warmup schedule, and ReduceLROnPlateau scheduler.


In [ ]:

# Allow TensorFloat32 for faster training on supported NVIDIA GPUs.
# TF32 can improve throughput on modern NVIDIA cards while keeping training stable.
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

criterion = LabelSmoothingCrossEntropy(
    smoothing=label_smoothing,
    ignore_index=-100  # no padding in fixed-length chunks, but kept for generality
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay
)

# Warmup uses the actual number of optimizer steps implied by the train loader.
total_steps = max(1, len(train_loader) * num_epochs)
warmup_steps = max(1, int(0.005 * total_steps))

warmup_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

plateau_scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=4,
    threshold=1e-4,
    min_lr=5e-7,
)

print(f"Optimizer initialized with lr = {optimizer.param_groups[0]['lr']:.8f}")
print(f"Total training steps = {total_steps}")
print(f"Warmup steps = {warmup_steps}")


## Section 34 — Hyperparameter Reminder Notes
A commented block that records one convenient hyperparameter snapshot for reference.


In [ ]:
# # Just so I can see it here without having to scroll up again
# # Initializing global settings
# vocab_size=16000
# bpe_min_frequency = 5
# block_size=256
# batch_size =32
# dim=256
# num_heads=4
# ff_dim=4*dim
# num_layers=4
# dropout=0.1
# val_split=0.1
# test_split=0.1
# num_epochs = 10000
# lr = 1e-4


## Section 35 — Training Call
This is the main training run. It trains, evaluates periodically, checkpoints progress, and updates the best model when validation improves.


In [ ]:
train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=num_epochs,
    train_mode="full",
    target_batches=len(train_loader),
    early_stop_patience=10,
    val_loss_threshold=val_loss_threshold,
    val_threshold_patience_evals=val_threshold_patience_evals,
    eval_every_n_epochs=eval_every_n_epochs,
    warmup_scheduler=warmup_scheduler,
    plateau_scheduler=plateau_scheduler,
    warmup_steps=warmup_steps,
    grad_clip_norm=grad_clip_norm,
    eval_max_batches=None,
)


## Section 36 — Informal Interpretation Guide
A rough, project-specific note about how to interpret validation loss values for this model and corpus. Treat this as intuition, not a universal rule.


### Val_Loss and it's meaning, roughly speaking:

- `val_loss > 8.0`: The model is guessing. It doesn't know anything. ---> Pure gibbereish
- `6.0 < val_loss < 8.0`: very basic understanding of sentence structure is learned. ---> starting to learn
- `4.5 < val_loss < 6.0`: sentences start to make sense. ---> novice understanding of syntax and sentence structure
- `2.5 < val_loss < 4.5`: Decent text generation ---> learning semantics and style
- `1.5 < val_loss < 2.5`: Really good story telling ---> higher-quality in text generation
- `val_loss < 1.5`: Either it is memorizing things very well, or it's god-like in story telling... hopefully the latter.

I can't get my val_loss under 4.5.... so the text is being generated... but it is awful.... and it's heavily skewed towards harry potter and lord of the rings.... I need better data... and more of it...

### Early-stopping logic used in this notebook

- `early_stop_patience` now counts **consecutive validation evaluations without improvement**.
- `val_threshold_patience_evals` now counts **consecutive validation evaluations** with `val_loss < val_loss_threshold`.
- This fixes the old mismatch where the code was labeled like it tracked evaluations/batches but actually counted epochs.


## Section 37 — Generation Stage
After training, the notebook switches from optimization to sampling and qualitative testing.


## Section 38 — Generation Utilities
This section defines helper functions for decoding generated text and for sampling from the trained model in a controlled way.


## Section 38 — Reload the Best Model
This reconstructs the model architecture and loads the best saved weights for evaluation/generation.


In [ ]:
# Reload the best saved model for generation/evaluation.
model = GPTModel(
    vocab_size=vocab_size,
    block_size=block_size,
    dim=dim,
    num_heads=num_heads,
    ff_dim=ff_dim,
    num_layers=num_layers,
    dropout=dropout,
).to(device)

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model from epoch {checkpoint.get('epoch', 'unknown')} with val_loss={checkpoint.get('val_loss', float('nan')):.4f}")


## Section 39 — Story Generation Function
This function samples text token-by-token using temperature, top-k / nucleus filtering, repetition penalty, and optional end-token stopping.


In [ ]:
def clean_generated_text(text):
    """Light post-processing to remove common spacing artifacts from decoding."""
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)   # remove spaces before punctuation
    text = re.sub(r"\(\s+", "(", text)              # remove spaces right after '('
    text = re.sub(r"\s+\)", ")", text)              # remove spaces right before ')'
    text = re.sub(r"[ 	]+", " ", text)              # collapse repeated spaces
    return text.strip()


def generate_story(
    model,
    tokenizer,
    prompt,
    max_new_tokens=500,
    temperature=1.0,
    top_N=None,
    top_p=None,
    repetition_penalty=1.05,
    device=device,
    ending_bias_strength=2.0,
    wrap_up_start_frac=0.8,
    stop_on_end_token=True,
):
    """
    Generate text token-by-token from a prompt.

    Key sampling controls:
    - temperature: softens or sharpens the next-token distribution.
    - top_N: keep only the top-k logits before sampling.
    - top_p: nucleus sampling; keep the smallest set of tokens whose cumulative probability exceeds p.
    - repetition_penalty: lightly discourages recently used tokens from repeating too aggressively.
    - ending_bias_strength / wrap_up_start_frac: gently encourage punctuation near the end of long generations.
    """
    model.eval()
    if temperature <= 0:
        raise ValueError("temperature must be > 0")

    input_ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long, device=device)
    generated = input_ids[:, -model.block_size:]

    end_token_id = tokenizer.token_to_id("[End]")
    ending_token_ids = [tok_id for tok_id in [tokenizer.token_to_id(sym) for sym in [".", "!", "?"]] if tok_id is not None]

    for step in range(max_new_tokens):
        inputs = generated[:, -model.block_size:]
        with torch.no_grad():
            logits = model(inputs)[:, -1, :]

        logits = logits / temperature

        # Repetition penalty on recently generated tokens.
        recent_tokens = generated[0, -64:].tolist()
        for token_id in set(recent_tokens):
            logits[:, token_id] /= repetition_penalty

        if top_N is not None:
            k = min(top_N, logits.size(-1))
            kth_values = torch.topk(logits, k, dim=-1).values[:, -1].unsqueeze(-1)
            logits = torch.where(logits < kth_values, torch.full_like(logits, float('-inf')), logits)

        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = False
            removal_mask = torch.zeros_like(logits, dtype=torch.bool)
            removal_mask.scatter_(1, sorted_indices, sorted_indices_to_remove)
            logits = logits.masked_fill(removal_mask, float('-inf'))

        # Near the end of long generations, make sentence-ending punctuation a bit more likely.
        if step >= int(wrap_up_start_frac * max_new_tokens):
            for token_id in ending_token_ids:
                logits[:, token_id] += ending_bias_strength

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat((generated, next_token), dim=1)

        if stop_on_end_token and end_token_id is not None and next_token.item() == end_token_id:
            break

    decoded = tokenizer.decode(generated[0].tolist())
    return clean_generated_text(decoded)


## Section 40 — Sample a Story
Provide a prompt, run generation, and print the decoded result.


In [ ]:
# Example generation prompt.
# Adjust the prompt text and sampling hyperparameters to test different outputs.
prompt = "During the cold night, "

story = generate_story(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=100,
    temperature=0.97,
    top_N=50,
    top_p=0.95,
    repetition_penalty=1.08,
)

print(story)


## Section 41 — Free Space
This final cell is empty and can be used for extra experiments, notes, or custom generation prompts.
